#Slenga normalizētājs

pielāgots `google/flan-t5-base`

In [ ]:
!pip install -q transformers datasets sentencepiece accelerate evaluate wandb regex bert-score


In [ ]:
import os
import shutil
import glob
import inspect
import re
import regex as emoji_regex
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from datasets import Dataset
from bert_score import BERTScorer
from transformers import (
    T5ForConditionalGeneration,
    T5Tokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    set_seed,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cuda


# Konfigurācija

In [ ]:
CONFIG = {
    # W&B
    "USE_WANDB": True,
    "WANDB_PROJECT": "slang-normaliser",
    "WANDB_ENTITY": "",
    "WANDB_API_KEY": "",
    "RUN_NAME": "",
    "RUN_NAME_INCLUDE_TIME": True,

    # Dati
    "BASE_URL": "https://raw.githubusercontent.com/GustsDoniks/slang-normalisation/main/datasets",
    "TRAIN_FILE": "allCommentsNormalised_train.csv",
    "EVAL_FILE": "allCommentsNormalised_eval.csv",
    "TEST_FILE": "allCommentsNormalised_test.csv",
    "HUMAN_TEST_FILE": "allCommentsNormalisedHuman_test.csv",
    "TRANSLATION_FILE": "translationShort.csv",
    "SOURCE_COL": "original_comment",
    "TARGET_COL": "normalized_comment",

    # Modelis
    "MODEL_NAME": "google/flan-t5-base",

    # Uzvedne un padomi
    "USE_HINTS": True,              # False = netiek padoti padomi apmācības laikā
    "REMOVE_EMOJI_ONLY_HINTS": False,
    "EMOJI_COMBO_MAX_LEN": 6,       # garākā emoji sekvence translationShort.csv
    "PROMPT_STYLE": "short",       # "short" / "long"
    "LOWERCASE_COMMENT": True,

    "MAX_INPUT_LEN": 256,
    "MAX_TARGET_LEN": 128,

    "SEED": 42,
    "DETERMINISTIC": True,
    "NUM_TRAIN_EPOCHS": 8,
    "LEARNING_RATE": 3e-05,
    "PER_DEVICE_TRAIN_BATCH_SIZE": 4,
    "PER_DEVICE_EVAL_BATCH_SIZE": 4,
    "WARMUP_RATIO": 0.05,
    "WEIGHT_DECAY": 0.01,
    "FP16": False,

    # BERTscore iekļaušanas apjoms apmācības zaudējumfunkcijā
    "SEMANTIC_ALPHA": 0.5,

    "USE_BERTSCORE": True,
    "BERTSCORE_MODEL_TYPE": "distilbert-base-uncased",
    "BERTSCORE_LANG": "en",
    "BERTSCORE_RESCALE_WITH_BASELINE": False,
    "BERTSCORE_BATCH_SIZE": 16,

    "SAVE_TOTAL_LIMIT": 2,
    "SAVE_ONLY_MODEL_CHECKPOINTS": True,
    "CLEANUP_AFTER_RUN": True,
    "KEEP_TRAINER_OUTPUT_AFTER_RUN": False,
    "KEEP_FINAL_MODEL_AFTER_RUN": False,
    "KEEP_WANDB_LOCAL_FILES": False,
    "DRY_RUN_CLEANUP": False,

    "OUTPUT_DIR": "./slang-normalizer-flan-t5-runs",
    "SAVE_FINAL_DIR": "./slang-normalizer-flan-t5-final-runs",
}


def _clean_name_part(value):
    value = str(value)
    value = value.replace("google/", "")
    value = value.replace(".", "p")
    value = value.replace("/", "-")
    value = value.replace("_", "-")
    value = value.replace(" ", "")
    return value


def make_run_name(config):
    hints = "hints" if config["USE_HINTS"] else "no-hints"
    fp16 = "fp16" if config["FP16"] else "fp32"

    parts = [
        _clean_name_part(config["MODEL_NAME"].split("/")[-1]),
        hints,
        f"p-{_clean_name_part(config['PROMPT_STYLE'])}",
        f"lr{_clean_name_part(config['LEARNING_RATE'])}",
        f"ep{config['NUM_TRAIN_EPOCHS']}",
        f"bs{config['PER_DEVICE_TRAIN_BATCH_SIZE']}",
        f"in{config['MAX_INPUT_LEN']}",
        f"alpha{_clean_name_part(config['SEMANTIC_ALPHA'])}",
        fp16,
    ]

    if config.get("RUN_NAME_INCLUDE_TIME", True):
        parts.append(time.strftime("%Y%m%d-%H%M%S"))

    return "_".join(parts)


if not CONFIG["RUN_NAME"]:
    CONFIG["RUN_NAME"] = make_run_name(CONFIG)

CONFIG["RUN_OUTPUT_DIR"] = os.path.join(CONFIG["OUTPUT_DIR"], CONFIG["RUN_NAME"])
CONFIG["RUN_SAVE_FINAL_DIR"] = os.path.join(CONFIG["SAVE_FINAL_DIR"], CONFIG["RUN_NAME"])

print("Run name:", CONFIG["RUN_NAME"])
print("W&B project:", CONFIG["WANDB_PROJECT"])
print("W&B entity:", CONFIG["WANDB_ENTITY"])
print("Output dir:", CONFIG["RUN_OUTPUT_DIR"])
print("BERTScore enabled:", CONFIG["USE_BERTSCORE"])
print("BERTScore model:", CONFIG["BERTSCORE_MODEL_TYPE"])


Run name: flan-t5-base_no-hints_p-short_lr1e-05_ep9_bs4_in256_alpha0p005_fp32_20260611-144508
W&B project: slang-normaliser
W&B entity: gusts-doniks-university-of-latvia
Output dir: ./slang-normalizer-flan-t5-runs/flan-t5-base_no-hints_p-short_lr1e-05_ep9_bs4_in256_alpha0p005_fp32_20260611-144508
BERTScore enabled: True
BERTScore model: distilbert-base-uncased


In [ ]:
if CONFIG["USE_WANDB"]:
    import wandb

    os.environ["WANDB_PROJECT"] = CONFIG["WANDB_PROJECT"]
    os.environ["WANDB_ENTITY"] = CONFIG["WANDB_ENTITY"]

    if CONFIG["WANDB_API_KEY"]:
        os.environ["WANDB_API_KEY"] = CONFIG["WANDB_API_KEY"]
        wandb.login(key=CONFIG["WANDB_API_KEY"])
    else:
        wandb.login()

    wandb.init(
        project=CONFIG["WANDB_PROJECT"],
        entity=CONFIG["WANDB_ENTITY"],
        name=CONFIG["RUN_NAME"],
        config=CONFIG,
        tags=[
            CONFIG["MODEL_NAME"].split("/")[-1],
            "hints" if CONFIG["USE_HINTS"] else "no-hints",
            f"prompt-{CONFIG['PROMPT_STYLE']}",
            f"alpha-{CONFIG['SEMANTIC_ALPHA']}",
            f"seed-{CONFIG['SEED']}",
        ],
    )

    print("W&B logging enabled.")
    print("Project:", CONFIG["WANDB_PROJECT"])
    print("Entity:", CONFIG["WANDB_ENTITY"])
    print("Run:", CONFIG["RUN_NAME"])
else:
    os.environ["WANDB_DISABLED"] = "true"

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


W&B logging enabled.
Project: slang-normaliser
Entity: gusts-doniks-university-of-latvia
Run: flan-t5-base_no-hints_p-short_lr1e-05_ep9_bs4_in256_alpha0p005_fp32_20260611-144508


In [ ]:
base = CONFIG["BASE_URL"]

TRAIN_CSV = f"{base}/{CONFIG['TRAIN_FILE']}"
EVAL_CSV = f"{base}/{CONFIG['EVAL_FILE']}"
TEST_CSV = f"{base}/{CONFIG['TEST_FILE']}"
HUMAN_TEST_CSV = f"{base}/{CONFIG['HUMAN_TEST_FILE']}"
TRANSLATION_CSV = f"{base}/{CONFIG['TRANSLATION_FILE']}"

train_df = pd.read_csv(TRAIN_CSV)
eval_df = pd.read_csv(EVAL_CSV)
test_df = pd.read_csv(TEST_CSV)
translations_df = pd.read_csv(TRANSLATION_CSV)

try:
    human_test_df = pd.read_csv(HUMAN_TEST_CSV)
except Exception:
    human_test_df = None

print("Train rows:", len(train_df))
print("Eval rows:", len(eval_df))
print("Test rows:", len(test_df))
print("Human test rows:", 0 if human_test_df is None else len(human_test_df))
print("Translation rows:", len(translations_df))

Train rows: 2506
Eval rows: 314
Test rows: 313
Human test rows: 70
Translation rows: 754


In [ ]:
SOURCE_COL = CONFIG["SOURCE_COL"]
TARGET_COL = CONFIG["TARGET_COL"]

assert SOURCE_COL in train_df.columns, f"Missing source column: {SOURCE_COL}"
assert TARGET_COL in train_df.columns, f"Missing target column: {TARGET_COL}"

def clean_split(df):
    df = df.copy()
    df = df.dropna(subset=[SOURCE_COL, TARGET_COL])
    df[SOURCE_COL] = df[SOURCE_COL].astype(str).str.strip()
    df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip()
    df = df[(df[SOURCE_COL] != "") & (df[TARGET_COL] != "")]
    return df.reset_index(drop=True)

train_df = clean_split(train_df)
eval_df = clean_split(eval_df)
test_df = clean_split(test_df)

if human_test_df is not None:
    human_test_df = clean_split(human_test_df)

translations_df.columns = [str(c).strip().lower() for c in translations_df.columns]
assert "term" in translations_df.columns, "translationShort.csv must contain a 'term' column"
assert "meaning" in translations_df.columns, "translationShort.csv must contain a 'meaning' column"

translations_df = translations_df.dropna(subset=["term", "meaning"])
translations_df["term"] = translations_df["term"].astype(str).str.strip().str.lower()
translations_df["meaning"] = translations_df["meaning"].astype(str).str.strip()
translations_df = translations_df[(translations_df["term"] != "") & (translations_df["meaning"] != "")]

def is_emoji_only_or_symbol_only(term):
    return re.search(r"[a-zA-Z0-9]", str(term)) is None

slang_dict = {}

for _, row in translations_df.iterrows():
    term = row["term"]
    meaning = row["meaning"]

    if CONFIG["REMOVE_EMOJI_ONLY_HINTS"] and is_emoji_only_or_symbol_only(term):
        continue

    slang_dict[term] = meaning

print("Clean train rows:", len(train_df))
print("Clean eval rows:", len(eval_df))
print("Clean test rows:", len(test_df))
print("Short slang hints loaded:", len(slang_dict))

Clean train rows: 2506
Clean eval rows: 314
Clean test rows: 313
Short slang hints loaded: 736


# Padomi



In [ ]:
def shorten_repeats_max2(text):
    # yaaassss -> yaass, brooo -> broo
    return re.sub(r"(.)\1{2,}", r"\1\1", str(text))


def shorten_repeats_max1(text):
    # yaaassss -> yas, brooo -> bro
    return re.sub(r"(.)\1+", r"\1", str(text))


EMOJI_CLUSTER_RE = emoji_regex.compile(r"\X")
EMOJI_DETECT_RE = emoji_regex.compile(r"\p{Extended_Pictographic}")


def is_emoji_cluster(cluster):
    return bool(EMOJI_DETECT_RE.search(str(cluster)))


def split_graphemes(text):
    return EMOJI_CLUSTER_RE.findall(str(text))


def separate_emojis_from_text(text):
    # "happy😂lol" -> "happy 😂 lol"
    pieces = []

    for cluster in split_graphemes(text):
        if is_emoji_cluster(cluster):
            pieces.append(f" {cluster} ")
        else:
            pieces.append(cluster)

    return re.sub(r"\s+", " ", "".join(pieces)).strip()


def get_lookup_variants(text):
    text = str(text).lower().strip()
    emoji_spaced = separate_emojis_from_text(text)

    variants = [
        text,
        emoji_spaced,
        shorten_repeats_max2(text),
        shorten_repeats_max2(emoji_spaced),
        shorten_repeats_max1(text),
        shorten_repeats_max1(emoji_spaced),
    ]

    return list(dict.fromkeys(variants))


def extract_emoji_sequences(text):
    # "wow 😂😂 ok 🔥❤️" -> [["😂", "😂"], ["🔥", "❤️"]]

    sequences = []
    current = []

    for cluster in split_graphemes(text):
        if is_emoji_cluster(cluster):
            current.append(cluster)
        else:
            if current:
                sequences.append(current)
                current = []

    if current:
        sequences.append(current)

    return sequences


def add_hint(hints, matched_terms, term, meaning):
    term = str(term).lower().strip()

    if term and term not in matched_terms:
        hints.append(f"{term}={meaning}")
        matched_terms.add(term)


def add_emoji_hints(text, slang_dict, hints, matched_terms):
    max_combo_len = int(CONFIG.get("EMOJI_COMBO_MAX_LEN", 6))

    for sequence in extract_emoji_sequences(text):
        i = 0

        while i < len(sequence):
            matched = False
            max_len = min(max_combo_len, len(sequence) - i)

            for combo_len in range(max_len, 0, -1):
                combo = "".join(sequence[i:i + combo_len]).lower()

                if combo in slang_dict:
                    add_hint(hints, matched_terms, combo, slang_dict[combo])
                    i += combo_len
                    matched = True
                    break

            if matched:
                while i < len(sequence) and i > 0 and sequence[i] == sequence[i - 1]:
                    i += 1
                continue

            i += 1


def find_hints(text, slang_dict):
    if not CONFIG["USE_HINTS"]:
        return []

    raw_text = str(text).lower().strip()
    text_variants = get_lookup_variants(raw_text)

    hints = []
    matched_terms = set()

    add_emoji_hints(raw_text, slang_dict, hints, matched_terms)

    sorted_terms = sorted(
        slang_dict.keys(),
        key=lambda term: (len(str(term).split()), len(str(term))),
        reverse=True,
    )

    for term in sorted_terms:
        term_clean = str(term).lower().strip()
        meaning = slang_dict[term]

        if not term_clean:
            continue

        if is_emoji_only_or_symbol_only(term_clean):
            continue

        pattern = r"(?<!\w)" + re.escape(term_clean) + r"(?!\w)"

        for variant_text in text_variants:
            if re.search(pattern, variant_text):
                add_hint(hints, matched_terms, term_clean, meaning)
                break

    return hints


In [ ]:
sample_for_hint_check = "so happy❤️😂😂"
print("Sample:", sample_for_hint_check)
print("Lookup variants:", get_lookup_variants(sample_for_hint_check))
print("Hints:", find_hints(sample_for_hint_check, slang_dict))

Sample: so happy❤️😂😂
Lookup variants: ['so happy❤️😂😂', 'so happy ❤️ 😂 😂', 'so hapy❤️😂', 'so hapy ❤️ 😂 😂']
Hints: []


# Uzvednes uzbūve

In [ ]:
def build_input(text):
    comment = str(text).strip()

    if CONFIG["LOWERCASE_COMMENT"]:
        comment = comment.lower()

    hints = find_hints(comment, slang_dict)

    if CONFIG["PROMPT_STYLE"] == "long":
        if hints:
            return (
                "Normalize the social media comment into clear standard English. "
                "Use the hints only to understand slang meanings. "
                "Do not explain or define slang. "
                "Do not copy the hint text as a definition. "
                "Return only the normalized comment. "
                f"Hints: {'; '.join(hints)}. "
                f"Comment: {comment}"
            )

        return (
            "Normalize the social media comment into clear standard English. "
            "Do not explain or define slang. "
            "Return only the normalized comment. "
            f"Comment: {comment}"
        )

    if CONFIG["PROMPT_STYLE"] == "short":
        if hints:
            return (
                "Rewrite the comment in clear standard English. "
                "Use slang hints only for meaning. "
                "Do not explain slang. "
                "Output only the rewritten comment. "
                f"Hints: {'; '.join(hints)}. "
                f"Comment: {comment}"
            )

        return (
            "Rewrite the comment in clear standard English. "
            "Do not explain slang. "
            "Output only the rewritten comment. "
            f"Comment: {comment}"
        )

    raise ValueError(f"Unknown PROMPT_STYLE: {CONFIG['PROMPT_STYLE']}")

for sample in [
    "Brooo this song is fireee frrrr",
    "Copying my post smh",
    "So happy to hear that ❤️",
]:
    print("=" * 100)
    print("Original:", sample)
    print("Hints:", find_hints(sample, slang_dict))
    print("Prompt:", build_input(sample))

Original: Brooo this song is fireee frrrr
Hints: []
Prompt: Rewrite the comment in clear standard English. Do not explain slang. Output only the rewritten comment. Comment: brooo this song is fireee frrrr
Original: Copying my post smh
Hints: []
Prompt: Rewrite the comment in clear standard English. Do not explain slang. Output only the rewritten comment. Comment: copying my post smh
Original: So happy to hear that ❤️
Hints: []
Prompt: Rewrite the comment in clear standard English. Do not explain slang. Output only the rewritten comment. Comment: so happy to hear that ❤️


In [ ]:
tokenizer = T5Tokenizer.from_pretrained(CONFIG["MODEL_NAME"])
model = T5ForConditionalGeneration.from_pretrained(CONFIG["MODEL_NAME"])
model = model.to(device)

print("Loaded:", CONFIG["MODEL_NAME"])

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded: google/flan-t5-base


# Dataframe -> HF Dataset

In [ ]:
def pandas_to_hf_dataset(df):
    input_texts = [build_input(x) for x in df[SOURCE_COL].tolist()]
    target_texts = df[TARGET_COL].astype(str).tolist()

    return Dataset.from_dict({
        "input_text": input_texts,
        "target_text": target_texts,
    })

train_text_ds = pandas_to_hf_dataset(train_df)
eval_text_ds = pandas_to_hf_dataset(eval_df)
test_text_ds = pandas_to_hf_dataset(test_df)

if human_test_df is not None:
    human_test_text_ds = pandas_to_hf_dataset(human_test_df)
else:
    human_test_text_ds = None

print(train_text_ds)
print(eval_text_ds)
print(test_text_ds)

Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 2506
})
Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 314
})
Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 313
})


In [ ]:
def tokenize(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=CONFIG["MAX_INPUT_LEN"],
        padding="max_length",
        truncation=True,
    )

    labels = tokenizer(
        batch["target_text"],
        max_length=CONFIG["MAX_TARGET_LEN"],
        padding="max_length",
        truncation=True,
    )

    cleaned_labels = []

    for label_ids in labels["input_ids"]:
        cleaned = [
            token_id if token_id != tokenizer.pad_token_id else -100
            for token_id in label_ids
        ]
        cleaned_labels.append(cleaned)

    model_inputs["labels"] = cleaned_labels
    return model_inputs

train_ds = train_text_ds.map(tokenize, batched=True, remove_columns=["input_text", "target_text"])
eval_ds = eval_text_ds.map(tokenize, batched=True, remove_columns=["input_text", "target_text"])
test_ds = test_text_ds.map(tokenize, batched=True, remove_columns=["input_text", "target_text"])

if human_test_text_ds is not None:
    human_test_ds = human_test_text_ds.map(tokenize, batched=True, remove_columns=["input_text", "target_text"])
else:
    human_test_ds = None

print(train_ds)
print(eval_ds)
print(test_ds)

Map:   0%|          | 0/2506 [00:00<?, ? examples/s]

Map:   0%|          | 0/314 [00:00<?, ? examples/s]

Map:   0%|          | 0/313 [00:00<?, ? examples/s]

Map:   0%|          | 0/70 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2506
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 314
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 313
})


# Apmācība ar semantisko līdzību zaudējumfunkcijā

Ja `SEMANTIC_ALPHA == 0`, šis strāda kā parasta Seq2Seq apmācība

In [ ]:
class SemanticSeq2SeqTrainer(Seq2SeqTrainer):
    def __init__(self, *args, semantic_alpha=0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.semantic_alpha = semantic_alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        outputs = model(**inputs)
        ce_loss = outputs.loss

        if self.semantic_alpha == 0.0:
            return (ce_loss, outputs) if return_outputs else ce_loss

        logits = outputs.logits
        embedding_matrix = model.shared.weight

        probs = torch.softmax(logits, dim=-1)
        pred_token_embeds = torch.matmul(probs, embedding_matrix)

        safe_labels = labels.clone()
        safe_labels[safe_labels == -100] = tokenizer.pad_token_id

        target_token_embeds = model.shared(safe_labels)
        mask = (labels != -100).unsqueeze(-1).float()

        pred_sentence_embed = (
            (pred_token_embeds * mask).sum(dim=1)
            / mask.sum(dim=1).clamp(min=1.0)
        )

        target_sentence_embed = (
            (target_token_embeds * mask).sum(dim=1)
            / mask.sum(dim=1).clamp(min=1.0)
        )

        cosine_sim = F.cosine_similarity(
            pred_sentence_embed,
            target_sentence_embed,
            dim=-1,
        )

        semantic_loss = 1 - cosine_sim.mean()
        total_loss = ce_loss + self.semantic_alpha * semantic_loss

        return (total_loss, outputs) if return_outputs else total_loss

In [ ]:
_BERT_SCORER = None


def get_bert_scorer():
    global _BERT_SCORER

    if _BERT_SCORER is None:
        print("Loading BERTScore model:", CONFIG["BERTSCORE_MODEL_TYPE"])
        _BERT_SCORER = BERTScorer(
            model_type=CONFIG["BERTSCORE_MODEL_TYPE"],
            lang=CONFIG["BERTSCORE_LANG"],
            rescale_with_baseline=CONFIG["BERTSCORE_RESCALE_WITH_BASELINE"],
            device=device,
        )

    return _BERT_SCORER


def sanitize_token_ids(token_ids):
    arr = np.asarray(token_ids)

    if arr.ndim == 3:
        arr = np.argmax(arr, axis=-1)

    arr = arr.astype(np.int64, copy=False)

    vocab_size = len(tokenizer)
    pad_id = tokenizer.pad_token_id

    invalid_mask = (arr < 0) | (arr >= vocab_size)
    arr = np.where(invalid_mask, pad_id, arr)

    return arr


def safe_batch_decode(token_ids):
    safe_ids = sanitize_token_ids(token_ids)

    return tokenizer.batch_decode(
        safe_ids,
        skip_special_tokens=True,
    )


def normalize_for_metric(texts):
    return [str(text).strip().lower() for text in texts]


def compute_text_metrics(pred_texts, label_texts):
    pred_texts = normalize_for_metric(pred_texts)
    label_texts = normalize_for_metric(label_texts)

    exact_match = np.mean([
        pred == label
        for pred, label in zip(pred_texts, label_texts)
    ])

    avg_pred_len = np.mean([len(pred.split()) for pred in pred_texts])
    avg_label_len = np.mean([len(label.split()) for label in label_texts])

    metrics = {
        "exact_match": float(exact_match),
        "avg_pred_len": float(avg_pred_len),
        "avg_label_len": float(avg_label_len),
    }

    if CONFIG["USE_BERTSCORE"]:
        bert_preds = [pred if pred else " " for pred in pred_texts]
        bert_labels = [label if label else " " for label in label_texts]

        scorer = get_bert_scorer()
        precision, recall, f1 = scorer.score(
            bert_preds,
            bert_labels,
            batch_size=CONFIG["BERTSCORE_BATCH_SIZE"],
        )

        metrics.update({
            "bertscore_precision": float(precision.mean().item()),
            "bertscore_recall": float(recall.mean().item()),
            "bertscore_f1": float(f1.mean().item()),
        })

    return metrics


def compute_metrics(eval_preds):
    preds, labels = eval_preds

    decoded_preds = safe_batch_decode(preds)
    decoded_labels = safe_batch_decode(labels)

    return compute_text_metrics(decoded_preds, decoded_labels)


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

optional_training_arg_kwargs = {}
if "save_only_model" in inspect.signature(Seq2SeqTrainingArguments.__init__).parameters:
    optional_training_arg_kwargs["save_only_model"] = CONFIG["SAVE_ONLY_MODEL_CHECKPOINTS"]

training_args = Seq2SeqTrainingArguments(
    output_dir=CONFIG["RUN_OUTPUT_DIR"],
    run_name=CONFIG["RUN_NAME"],

    num_train_epochs=CONFIG["NUM_TRAIN_EPOCHS"],
    learning_rate=CONFIG["LEARNING_RATE"],

    per_device_train_batch_size=CONFIG["PER_DEVICE_TRAIN_BATCH_SIZE"],
    per_device_eval_batch_size=CONFIG["PER_DEVICE_EVAL_BATCH_SIZE"],
    warmup_ratio=CONFIG["WARMUP_RATIO"],
    weight_decay=CONFIG["WEIGHT_DECAY"],

    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=CONFIG["SAVE_TOTAL_LIMIT"],

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    predict_with_generate=True,
    generation_max_length=CONFIG["MAX_TARGET_LEN"],
    fp16=CONFIG["FP16"],

    seed=CONFIG["SEED"],
    data_seed=CONFIG["SEED"],

    report_to="wandb" if CONFIG["USE_WANDB"] else "none",
    **optional_training_arg_kwargs,
)

trainer = SemanticSeq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    semantic_alpha=CONFIG["SEMANTIC_ALPHA"],
)

print("Trainer ready.")
print("Run name:", CONFIG["RUN_NAME"])
print("W&B project:", CONFIG["WANDB_PROJECT"])
print("W&B entity:", CONFIG["WANDB_ENTITY"])
print("Output dir:", CONFIG["RUN_OUTPUT_DIR"])
print("Semantic alpha:", CONFIG["SEMANTIC_ALPHA"])
print("Save total limit:", CONFIG["SAVE_TOTAL_LIMIT"])
print("Optional training args:", optional_training_arg_kwargs)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer ready.
Run name: flan-t5-base_no-hints_p-short_lr1e-05_ep9_bs4_in256_alpha0p005_fp32_20260611-144508
W&B project: slang-normaliser
W&B entity: gusts-doniks-university-of-latvia
Output dir: ./slang-normalizer-flan-t5-runs/flan-t5-base_no-hints_p-short_lr1e-05_ep9_bs4_in256_alpha0p005_fp32_20260611-144508
Semantic alpha: 0.005


In [ ]:
trainer.train()

Starting training...


Epoch,Training Loss,Validation Loss,Exact Match,Avg Pred Len,Avg Label Len,Bertscore Precision,Bertscore Recall,Bertscore F1
1,1.672967,1.578957,0.000000,10.286624,12.203822,0.875963,0.860366,0.867719
2,1.587016,1.404083,0.009554,10.729299,12.203822,0.882675,0.869824,0.875800
3,1.455638,1.320020,0.015924,10.939490,12.203822,0.886836,0.875180,0.880591
4,1.400435,1.266470,0.019108,11.025478,12.203822,0.891974,0.881305,0.886190
5,1.441627,1.230409,0.022293,10.996815,12.203822,0.894109,0.882546,0.887874
6,1.226539,1.204235,0.022293,11.066879,12.203822,0.896556,0.885398,0.890536
7,1.285004,1.186932,0.022293,11.006369,12.203822,0.898359,0.886571,0.892028
8,1.240721,1.178404,0.022293,10.990446,12.203822,0.898700,0.886631,0.892233
9,1.270933,1.177405,0.022293,10.980892,12.203822,0.898782,0.886831,0.892375


Loading BERTScore model: distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Training complete.


# Modeļa saglabāšana

In [ ]:
model.save_pretrained(CONFIG["RUN_SAVE_FINAL_DIR"])
tokenizer.save_pretrained(CONFIG["RUN_SAVE_FINAL_DIR"])

print("Saved final model to:", CONFIG["RUN_SAVE_FINAL_DIR"])

if CONFIG["USE_WANDB"]:
    import wandb
    wandb.save(os.path.join(CONFIG["RUN_SAVE_FINAL_DIR"], "*"))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wandb: WARNING Symlinked 5 files into the W&B run directory; call wandb.save again to sync new files.


Saved final model to: ./slang-normalizer-flan-t5-final-runs/flan-t5-base_no-hints_p-short_lr1e-05_ep9_bs4_in256_alpha0p005_fp32_20260611-144508


In [ ]:
model.eval()


def normalize(text):
    """Normalize one comment using the same prompt-building logic as training/evaluation."""
    prompt = build_input(text)

    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=CONFIG["MAX_INPUT_LEN"],
        truncation=True,
    )

    input_ids = encoded["input_ids"].to(model.device)
    attention_mask = encoded["attention_mask"].to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=CONFIG["MAX_TARGET_LEN"],
        )

    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
    ).strip()


examples = [
    "based",
    "w",
    "rare uk w",
    "true asf",
    "frrrr",
    "brooo this song is fireee frrrr",
    "the house is on fire",
    "i bought bread",
    "he got his bread up",
    "bro fell off",
    "this album dropped",
]

for ex in examples:
    print("=" * 100)
    print("Input:", ex)
    print("Output:", normalize(ex))


Input: based
Output: Based on that.
Input: w
Output: W.
Input: rare uk w
Output: Rare UK win.
Input: true asf
Output: True, as if.
Input: frrrr
Output: For real.
Input: brooo this song is fireee frrrr
Output: Bro, this song is really good, for real.
Input: the house is on fire
Output: The house is on fire.
Input: i bought bread
Output: I bought bread.
Input: he got his bread up
Output: He got his bread up.
Input: bro fell off
Output: He fell off.
Input: this album dropped
Output: This album has been very successful.


In [ ]:
def predict_dataframe(df, output_csv, max_rows=None):
    rows = df if max_rows is None else df.head(max_rows)
    predictions = []

    for _, row in rows.iterrows():
        source = row[SOURCE_COL]
        target = row[TARGET_COL]
        pred = normalize(source)

        predictions.append({
            "original_comment": source,
            "target_normalization": target,
            "prediction": pred,
        })

    pred_df = pd.DataFrame(predictions)
    pred_df.to_csv(output_csv, index=False)

    print("Saved:", output_csv)
    return pred_df


def score_prediction_dataframe(pred_df, prefix):
    metrics = compute_text_metrics(
        pred_df["prediction"].tolist(),
        pred_df["target_normalization"].tolist(),
    )

    prefixed_metrics = {
        f"{prefix}_{key}": value
        for key, value in metrics.items()
    }

    print(prefix, "metrics:")
    for key, value in prefixed_metrics.items():
        print(key, value)

    if CONFIG["USE_WANDB"]:
        import wandb
        wandb.log(prefixed_metrics)

    return prefixed_metrics


full_test_csv = f"flan_t5_full_test_predictions_{CONFIG['RUN_NAME']}.csv"
human_test_csv = f"flan_t5_human_test_predictions_{CONFIG['RUN_NAME']}.csv"

full_test_predictions = predict_dataframe(
    test_df,
    full_test_csv,
)

full_test_metrics = score_prediction_dataframe(
    full_test_predictions,
    "full_test",
)

if human_test_df is not None:
    human_test_predictions = predict_dataframe(
        human_test_df,
        human_test_csv,
    )

    human_test_metrics = score_prediction_dataframe(
        human_test_predictions,
        "human_test",
    )
else:
    human_test_predictions = None
    human_test_metrics = None

if CONFIG["USE_WANDB"]:
    import wandb
    wandb.save(full_test_csv)
    if human_test_predictions is not None:
        wandb.save(human_test_csv)


Saved: flan_t5_full_test_predictions_flan-t5-base_no-hints_p-short_lr1e-05_ep9_bs4_in256_alpha0p005_fp32_20260611-144508.csv
full_test metrics:
full_test_exact_match 0.025559105431309903
full_test_avg_pred_len 11.217252396166135
full_test_avg_label_len 12.654952076677317
full_test_bertscore_precision 0.8991943001747131
full_test_bertscore_recall 0.885242223739624
full_test_bertscore_f1 0.891700804233551
Saved: flan_t5_human_test_predictions_flan-t5-base_no-hints_p-short_lr1e-05_ep9_bs4_in256_alpha0p005_fp32_20260611-144508.csv


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


human_test metrics:
human_test_exact_match 0.02857142857142857
human_test_avg_pred_len 11.742857142857142
human_test_avg_label_len 14.1
human_test_bertscore_precision 0.881763756275177
human_test_bertscore_recall 0.8693050146102905
human_test_bertscore_f1 0.8745201230049133


In [ ]:
if CONFIG["USE_WANDB"]:
    import wandb
    wandb.finish()

eval/avg_label_len,▁▁▁▁▁▁▁▁▁
eval/avg_pred_len,▁▅▇█▇█▇▇▇
eval/bertscore_f1,▁▃▅▆▇▇███
eval/bertscore_precision,▁▃▄▆▇▇███
eval/bertscore_recall,▁▄▅▇▇████
eval/exact_match,▁▄▆▇█████
eval/loss,█▅▃▃▂▁▁▁▁
eval/runtime,█▇▃█▂▂▂▃▁
eval/samples_per_second,▁▂▆▁▇▇▇▆█
eval/steps_per_second,▁▂▆▁▇▇▇▆█
+17,...


W&B run finished.


In [ ]:
def _folder_size_gb(path):
    total = 0
    if not os.path.exists(path):
        return 0.0
    for root, _, files in os.walk(path):
        for name in files:
            fp = os.path.join(root, name)
            try:
                total += os.path.getsize(fp)
            except OSError:
                pass
    return total / (1024 ** 3)


def _remove_path(path, label):
    if not path or not os.path.exists(path):
        print(f"Skip {label}: not found -> {path}")
        return

    size_gb = _folder_size_gb(path) if os.path.isdir(path) else os.path.getsize(path) / (1024 ** 3)
    print(f"{'Would delete' if CONFIG['DRY_RUN_CLEANUP'] else 'Deleting'} {label}: {path} ({size_gb:.2f} GB)")

    if not CONFIG["DRY_RUN_CLEANUP"]:
        if os.path.isdir(path):
            shutil.rmtree(path, ignore_errors=True)
        else:
            try:
                os.remove(path)
            except OSError as exc:
                print(f"Could not delete {path}: {exc}")


def cleanup_after_run():
    if not CONFIG["CLEANUP_AFTER_RUN"]:
        print("Cleanup disabled.")
        return

    print("Cleanup starting...")

    if not CONFIG["KEEP_TRAINER_OUTPUT_AFTER_RUN"]:
        _remove_path(CONFIG["RUN_OUTPUT_DIR"], "trainer output/checkpoints")

    if not CONFIG["KEEP_FINAL_MODEL_AFTER_RUN"]:
        _remove_path(CONFIG["RUN_SAVE_FINAL_DIR"], "saved final model")

    if not CONFIG["KEEP_WANDB_LOCAL_FILES"]:
        _remove_path("./wandb", "local W&B cache")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Cleanup complete.")


cleanup_after_run()
